# Run Match

In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100`% !important; }</style>"))
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [ ]:
from timeutils import Timestat
from master import MasterParams, MasterDBs, MasterPaths, MasterMetas, MusicDBPermDir
from musicdb import PanDBIO
from gate import MusicDBGate
from pandas import DataFrame, Series, concat
import musicdb
from ioutils import HTMLIO, FileIO
from listUtils import getFlatList
import swifter
import dask.dataframe as dd
from match import MatchDBDataIO, AlbumReq, NameReq, MatchReq, MatchDB, CrossMatchDB, PanDBMatch
io = FileIO()

In [ ]:
baseReqs = {"MetalArchives": 4,
            "RateYourMusic": 20,
            "Beatport": 35,
            "Spotify": 20,
            "Discogs": 3,  ## 12
            "Traxsource": 100000}
#baseDB    = "Beatport"
#baseDB    = "Discogs"
#baseDB    = "Spotify"
#baseDB    = "Traxsource"
#baseDB    = "MyMixTapez"
#baseDB    = "Genius"
#baseDB    = "MetalArchives"
baseDB    = "RateYourMusic"  # 3

minL = 1
maxL = 6

minA = 1
maxA = 30000000

#baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1))}
baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=minA, max=maxA))}
#baseReq   = {baseDB: AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1)}
#baseReq   = {baseDB: AlbumReq(min=10, max=baseReqs.get(baseDB,10000)+1, rnd=10000)}

compareDBs  = ["RateYourMusic", "LastFM", "Spotify", "Genius", "Discogs", "MusicBrainz", "Deezer", "MetalArchives"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "Beatport"] # "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Discogs", "MusicBrainz", "Beatport", "Traxsource", "Genius", "MyMixTapez", "MetalArchives"] # "LastFM", "Deezer"]
#compareDBs  = ["RateYourMusic", "Traxsource", "Spotify", "Beatport"]
compareReqs = {compareDB: MatchReq(NameReq(min=minL-5, max=maxL+5), AlbumReq(min=3)) for compareDB in compareDBs if compareDB not in [baseDB]}
compareDBs  = list(compareReqs.keys())

matchReqs  = {**baseReq, **compareReqs}
mediaTypes = ["Album", "SingleEP"]
mediaTypes = ["{0}Media".format(mediaType) for mediaType in mediaTypes]
mediaTypes = list(MasterMetas().getMedias().values())
reqs       = {"Media": mediaTypes, "Reqs": matchReqs, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Primary Run Params:")
print("  ==> DBs:   [{0}] x {1}]".format(baseDB,list(compareReqs.keys())))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(reqs["Match"]))

crossreqs  = {"Media": mediaTypes, "Reqs": {baseDB: MatchReq(AlbumReq(min=2))}, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Cross Match Run Params:")
print("  ==> DBs:   {0} x [{1}]".format(list(compareReqs.keys()), baseDB))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(crossreqs["Match"]))

In [ ]:
baseIO = MatchDBDataIO(db=baseDB, mediaTypes=reqs["Media"], mask=reqs["Mask"], verbose=True, base=True)
baseIO.loadNames()
baseIO.setAvailableNames(reqs["Reqs"][baseDB])
del baseIO

# Match & Cross Match

In [ ]:
mdb = MatchDB(baseDB=baseDB, compareDBs=compareDBs, reqs=reqs)
mdb.match()
mdb.save()
del mdb

In [ ]:
mdbpd = MusicDBPermDir()
mres  = io.get(mdbpd.getMatchPermPath().join("primaryMatch.p"))
cmdb  = CrossMatchDB(baseDB, mres, crossreqs, verbose=True)
cmdb.match()
cmdb.save()

del cmdb

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5, minName=5)
pdbm.master()
pdbm.merge(allownew=True, verbose=False)

In [ ]:
pdbm.mergeMultiRows()

# Extra Match

In [ ]:
# 
#043acf9148   | Discogs        572788                                    3   7    | FRANK NITTY                             FRANK ITT
#6685833dd1   | Discogs        203282                                    4   5    | WIL WHEATON                             WILL WHEATON
#4a6e7fa4a1   | MetalArchives  35881                                     3   7    | MONOLITHS                               MONOLITH
#f9aabd5eb2   | Discogs        476975                                    3   5    | PHIL AUSTIN                             PHILIP AUSTIN

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5, minName=3, maxName=5)

In [ ]:
pdbm.include("""
727a55b876   | Discogs        4241968                                   3   8    | VOHN BEATZ                              VOHNBEATZ
c6931db495   | Discogs        1155983                                   3   8    | SNYP LIFE                               SNYPLIFE
83d01f3676   | Spotify        3ndmfo71A1Wt3a3hOrmouL                    3   8    | BUBBLEGUM                               BUBBLE GUM
5d27ddbd82   | Discogs        3700594                                   3   8    | COLD FRONT                              COLDFRONT
eab8bb2d09   | Discogs        772027                                    4   6    | NIGHTHAWKS                              NIGHT HAWKS
20dbe919df   | Discogs        22689                                     4   8    | WAVESLAVES                              WAVE SLAVES
53a094de18   | Discogs        3672462                                   3   8    | اعجوبه ها                               اعجوبهها
cee6848c3e   | Discogs        272031                                    3   8    | ARTO-NETO                               ARTO - NETO
d8477d97e2   | Discogs        1345446                                   3   7    | THE 12 AM                               THE 12 A.M.
aac68c03ff   | Discogs        4817773                                   3   6    | BROTHERSU                               BROTHER SU
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=4, maxQual=5, minName=4)

In [ ]:
pdbm.include("""
a9ba802d0a   |                                           Genius         453256                                    5   4    | HANNE KROGH                             HANNE KROGH
eeb23aafac   |                                           Traxsource     1144                                      5   4    | TONY THOMAS                             TONY THOMAS
7fc91c0b7e   |                                           Genius         586768                                    5   4    | ANDY STURMER                            ANDY STURMER
c9b4749fe8   |                                           Spotify        0ZZ6jQHS2D5eDfyjqxXWmb                    5   4    | MUNNI BEGUM                             MUNNI BEGUM
d5c7139292   |                                           Discogs        511601                                    5   4    | LYNN KELLOGG                            LYNN KELLOGG
9bca3134a5   |                                           Genius         480715                                    5   4    | ELLA JENKINS                            ELLA JENKINS
f821f98f7a   |                                           Spotify        1uiij18oOdEQGuKasqf2XT                    5   4    | ALEXIS SMITH                            ALEXIS SMITH
9aab8575f3   |                                           Discogs        3355096                                   5   4    | DUSTY BROOKS                            DUSTY BROOKS
9cbb2a8486   |                                           Spotify        1EyJASmsQrxwWAD0uVbMvY                    5   4    | THE HORNETS                             THE HORNETS
e6a3108c6e   |                                           Spotify        1IB6GqZXvYSI7E2LVR4laZ                    5   4    | HANNA-MCEUEN                            HANNA-MCEUEN
66b0aa61f9   |                                           Discogs        858729                                    5   4    | LARRY HOPPEN                            LARRY HOPPEN
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=2, maxQual=4, minName=3)

In [ ]:
pdbm.include("""
77be3cee4d   |                                           Traxsource     24596                                     5   2    | NASTY HABITS                            NASTY HABITS
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=1, maxQual=2, minName=5)

In [ ]:
pdbm.include("""
4907b7a858   |                                           Discogs        266616                                    5   1    | GRUPPENBILD                             GRUPPENBILD
41f394b205   |                                           Beatport       41810                                     5   1    | SASCHA DIVE                             SASCHA DIVE
4f84193331   |                                           Spotify        67XOATo1PsPRHr2LvSM03z                    5   1    | THE DARNELLS                            THE DARNELLS
a65d3dc166   |                                           Beatport       62430                                     5   1    | IAN ASTBURY                             IAN ASTBURY
70dd9d0ccd   |                                           Discogs        389619                                    5   1    | BRITT EKLAND                            BRITT EKLAND
9eec3c346a   |                                           Spotify        1OEjS3V4nB7wmjP2YAXZI2                    5   1    | SHA MONEY XL                            SHA MONEY XL
f42923820c   |                                           Discogs        285528                                    5   1    | HOWIE CASEY                             HOWIE CASEY
""")

pdbm.master()
pdbm.merge()

In [ ]:
del pdbm

In [ ]:
# Fix This:
#afc404588f   | 183166                   Discogs        113700                                  2    STEVE MASTERSON                                   STEVE MAESTRO                                      | afc404588f
#a6da8c9ee9   | 25875                    Discogs        678236                                  4    ALTAR OF FLESH                                    ALTAR OF FLIES                                     | a6da8c9ee9

# New Matching Code

# New Single Matching Code

In [ ]:
names = smdb.baseIO.getAvailableNames()

In [ ]:
smdb = SingleMatchDB(baseDB="RateYourMusic", compareDB="Spotify", reqs=reqs)
smdb.match()
smdb.save()
del smdb


mdbpd = MusicDBPermDir()
mres  = io.get(mdbpd.getMatchPermPath().join("primaryMatch.p"))
scmdb = SingleCrossMatchDB(baseDB, mres, crossreqs, verbose=True)
scmdb.match()
scmdb.save()
del scmdb


In [ ]:
pdbio = PanDBIO()
mmeDF = pdbio.getData()

In [ ]:
mmeDF[mmeDF["RateYourMusic"] == '106836']

In [ ]:
mmeDF[mmeDF["Spotify"] == '3lk3F4u5qqxq8zFjwNj5U1']

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5)
pdbm.masterSingle()
#pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=4, maxQual=5)

In [ ]:
pdbm.include("""
1d2402a17a   | 1061693                  Spotify        49D8h67pxvvUNGOLKEGjkx                  4    OWAIN ARWEL HUGHES                                OWAIN ARWEL HUGHES                                 | 1d2402a17a
a86b2ef789   | 121809                   Spotify        6NSIW1uEq8JZmxEkHMF17c                  4    ANNA TOMOWA-SINTOW                                ANNA TOMOWA-SINTOW                                 | a86b2ef789
877d262f5e   | 142182                   Spotify        5DwQvVHPVspRvStEAN722N                  4    TAKÁCS QUARTET                                   TAKÁCS QUARTET                                    | 877d262f5e
a2f65f8447   | 412578                   Spotify        50skve7Y0al39yGqLuCMNu                  4    MAURICE ABRAVANEL                                 MAURICE ABRAVANEL                                  | a2f65f8447
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=2, maxQual=4)

In [ ]:
pdbm.include("""
9554709d72   | 405351                   Spotify        2mHCS8PPaV7cAmT3ew8qHY                  2    SAULIUS SONDECKIS                                 SAULIUS SONDECKIS                                  | 9554709d72
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=1, maxQual=2)

In [ ]:
pdbm.include("""
3178f6847b   | 337551                   Spotify        7N0fh2csz0eFkrE01LF1m3                  1    STRATOS PAGIOUMTZIS                               STRATOS PAGIOUMTZIS                                | 3178f6847b
842d333cee   | 375588                   Spotify        2LqWWIvCBaetjLStxk1XK6                  1    VAN AND SCHENCK                                   VAN & SCHENCK                                      | 842d333cee
60cc9bc61a   | 77193                    Spotify        6VeTIJi6Dlx8ywPfIwqALY                  1    ALBERT NICHOLAS                                   ALBERT NICHOLAS                                    | 60cc9bc61a
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)